In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
# This is just for the understanding doc contains 2 components ie page_content and metadata
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [4]:
type(sample_doc)

langchain_core.documents.base.Document

In [5]:
# Text data loading
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/python.txt", encoding="utf-8")

In [29]:
document = loader.load()
document

In [7]:
# PDF data loading 
from langchain_community.document_loaders.pdf import PyMuPDFLoader

pdf_loader = PyMuPDFLoader("data/research.pdf")

In [28]:
document = pdf_loader.load()
document

# Data ingestion pipeline

In [9]:
# Remember for this file structure pdfs(research1 and research2) are the vector db we will create 

In [10]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

#### Data converting into documents

In [11]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for file in os.listdir(folder_path):
        if file.lower().endswith(".pdf"):
            # complete filepath 
            pdf_path = os.path.join(folder_path, file)

            loader = PyPDFLoader(pdf_path)
            document = loader.load()

            all_docs.extend(document)
            num_docs += 1

    print("Total pdfs: ", num_docs)
    print("Total pages: ", len(all_docs))
    return all_docs

In [12]:
all_pdf_documents = load_all_pdfs()

Total pdfs:  2
Total pages:  41


#### Chunks

In [13]:
# !pip install langchain_text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [14]:
def split_docs(documents, chunk_size=500, chunk_overlap=50):
    test_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    chunked_docs = test_splitter.split_documents(documents)
    return chunked_docs

In [30]:
chunks = split_docs(all_pdf_documents)
chunks

In [16]:
len(chunks)

324

#### Embedding

In [17]:
from sentence_transformers import SentenceTransformer

In [18]:
# Creating a class do that we can make objects of it later
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name = model_name
        print("Loading Model: ", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("Embedding Dimensions: ", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("Embeddings shape: ", embeddings.shape)
        return embeddings

In [19]:
embedding_manager = EmbeddingManager()

Loading Model:  all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Dimensions:  384


#### Vector DB

In [20]:
# imp we will create vector store ie nothing but a interface used to interact with vector db
import chromadb
import uuid

In [56]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        
        # create a collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"Description": "Vector store collection for pdf embeddings in RAG"}
        )
        print("Initialized the vector store with collection: ", self.collection_name)
        print("Docs in collection: ", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents does not match number of embeddings")

        # store => ids, embeddings, documents, metadata
        ids=[]
        all_metadata=[]
        documents_content=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            # Add all in the collection
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )
        print("Total documents added in vector store: ", len(documents_content))
        print("Docs in collection: ", self.collection.count())

In [57]:
vector_store = VectorStoreManager()

Initialized the vector store with collection:  pdf_documents
Docs in collection:  0


In [58]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embeddings shape:  (324, 384)
Total documents added in vector store:  324
Docs in collection:  324


# Retrieval pipeline

In [59]:
from sklearn.metrics.pairwise import cosine_similarity

In [60]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]
        
        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs = []

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance # this is formula

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "metadata": metadata,
                        "document": document,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })
            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("No documents found")

        return retrieved_docs

In [61]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [63]:
rag_retriever.retrieve("What is RAG")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape:  (1, 384)
retrieved 5 documents


[{'id': 'doc_3ba97ab7-17a5-4fbb-84cd-e110e5f5aec1',
  'metadata': {'moddate': '2024-03-28T00:54:45+00:00',
   'author': '',
   'producer': 'pdfTeX-1.40.25',
   'trapped': '/False',
   'creator': 'LaTeX with hyperref',
   'page_label': '1',
   'total_pages': 21,
   'title': '',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'keywords': '',
   'page': 0,
   'subject': '',
   'doc_index': 91,
   'source': 'data/pdfs\\research2.pdf',
   'content_length': 288,
   'creationdate': '2024-03-28T00:54:45+00:00'},
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'distance': 0.4226757287979126,
  'similarity_score': 0.5773242712020874,
  'rank': 1},
 {'id': 'doc_73d638c8-